# CSD Data Manipulation - Category System

## How to Add New Categories

### Step 1: Update your tariff-hscodes CSV
Add entries with your new category names:
```csv
HS Code,Category
440710,Lumber (old)
440910,Lumber (new)
...
```

### Step 2: Update CATEGORY_CONFIG (Cell 3)
Map each CSV category name to a short column prefix:
```python
CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Lumber (old)': 'LumOld',   # <-- Add new categories
    'Lumber (new)': 'LumNew',   # <-- Add new categories
    ...
}
```

### Step 3: Run all cells
The notebook will automatically:
- Create NAICS sets for each category
- Generate `{Prefix}_B`, `{Prefix}_E` columns (weighted business/employee counts)
- Generate `{Prefix}_1`, `{Prefix}_2`, `{Prefix}_3` columns (percentages for choropleth)
- Export everything to GeoJSON, Shapefile, and CSV

### Output Columns per Category
- `{Prefix}_B` = Weighted number of businesses
- `{Prefix}_E` = Weighted number of employees (by work location)
- `{Prefix}_C` = Weighted number of employees (by residence)
- `{Prefix}_1` = % of all businesses affected
- `{Prefix}_2` = % of all employees affected (by work location)
- `{Prefix}_3` = % of census population in affected jobs (by residence)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union
from tqdm import tqdm
import json
import gc
import time
from datetime import timedelta

## Configuration: Define your tariff categories here

In [ ]:
# ============================================================
# CATEGORY CONFIGURATION - Edit this to add/modify categories
# ============================================================

# Path to the tariff-hscodes CSV file
TARIFF_HSCODES_FILE = '../raw/tariff_hs_codes_8_27_2026.csv'

# Category configuration: maps CSV category names to short column prefixes
# Format: 'Category Name in CSV': 'Short_Prefix_for_Columns'

CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Steel': 'Steel',
    'Copper': 'Cop',
    'Energy Mineral': 'Ene',
    'MHDV': 'MHDV',
    # Add new categories here:
    'Lumber (old)': 'LumOld',
    'Lumber (new)': 'LumNew',
    'Dairy': 'Dairy',
    'Alcohol': 'Alcohol',
    'Motor': 'Motor',
    'before August 22': 'before August 22',
    'after August 22': 'after August 22',
    'Section 338': 'Section 338'
}

# Always-included special categories (don't change these unless you know what you're doing)
SPECIAL_CATEGORIES = ['nonCUSMA', 'Total']  # nonCUSMA = goods not covered by CUSMA

print(f"✅ Configured {len(CATEGORY_CONFIG)} tariff categories: {list(CATEGORY_CONFIG.keys())}")
print(f"   Using tariff file: {TARIFF_HSCODES_FILE}")

# STEP 3: Connecting Tariffed HS Codes, NAICS Codes and respective CUSMA Non-Utilisation Rates via Concordance Table

In [ ]:
tariffed = pd.read_csv(TARIFF_HSCODES_FILE, encoding_errors='ignore', dtype={'HS Code': str})

tariffed['HS_Code_6digit'] = (
    tariffed['HS Code']
    .str.replace('.', '', regex=False)
    .str[:6]
)

tariffed = tariffed[['HS_Code_6digit', 'Category']].drop_duplicates()

# Validate that all categories in CSV are in our config
csv_categories = set(tariffed['Category'].dropna().unique())
configured_categories = set(CATEGORY_CONFIG.keys())
unknown_categories = csv_categories - configured_categories

if unknown_categories:
    print(f"⚠️ WARNING: Found categories in CSV not in CATEGORY_CONFIG: {unknown_categories}")
    print("   Add them to CATEGORY_CONFIG or they will be treated as 'nonCUSMA'")
else:
    print(f"✅ All CSV categories are configured: {csv_categories}")

In [ ]:
concordance = pd.read_csv('../raw/C616_HS8toNaics6_concord_202505.csv', dtype={'hts10': str})

concordance['HS_Code_6digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:6]
)

concordance['HS_Code_2digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:2]
)

concordance['NAICS'] = concordance['NAICS 6 Code'].astype(str)

concordance = concordance[['HS_Code_6digit', 'NAICS', 'HS_Code_2digit']].drop_duplicates()

print(concordance[concordance['HS_Code_6digit'] == '440311'].head())

In [ ]:
util = pd.read_csv('../raw/USMCA Utilization Data.csv')

util['HS_Code_2digit'] = (
    util['HS Classification']
    .astype(str)
    .str[:2]
)

util['nonutil_rate'] = util['USMCA_Nonutilisation_May2025']

util = util[['HS_Code_2digit', 'nonutil_rate']]

Setting the non-utilisation rate for those with sectoral tariffs at 1 reflects the fact that the 35% tariffs on non-CUSMA goods does not apply to the sectoral tariffs and that producers impacted by sectoral tariffs cannot use CUSMA to get their goods tariff-free.

In [ ]:
naics_tar = concordance.merge(tariffed, on='HS_Code_6digit', how='left')

counts = naics_tar['HS_Code_6digit'].value_counts().reset_index()
naics_tarc = naics_tar.merge(counts, on='HS_Code_6digit', how='left')

naics_imp = naics_tarc.merge(util, on='HS_Code_2digit', how='left')

# new column non-util chapter getting the nonutil rate! (for before section 338)
naics_imp['nonutil_chapter'] = naics_imp['nonutil_rate']

# Making the codes with a sectoral tariff category assigned to have a non-util rate value of 1
# This ensures that when multiplied later, sectoral tariffs do not affect the CUSMA non-utilisation rates to compute the impact of nonCUSMA 35% tariffs
naics_imp.loc[naics_imp['Category'].notna(), 'nonutil_rate'] = 1
naics_imp['Category'] = naics_imp['Category'].fillna('nonCUSMA')
naics_imp

# STEP 4: Getting Weights by Province/Territory

Since multiple NAICS codes may contribute to the production of one HS code product, and we do not how much of a part does each NAICS contribute to the whole HS code good production, **thus an assumption is made to divide them equally**. Hence, when each export value is added, it is divided them by the count (how many times does that HS Code get repeated).  

Meanwhile, the non-utilisation rate of CUSMA exemption by each HS Code is first multiplied to the total value of each HS Code export to the US, before divided by the count.

In [ ]:
EXPORT_DIR = '../raw/exports'

# Initialize the DataFrame with NAICS data
tariff_exp_val = naics_imp.copy()

# Define all regions to process
provinces = ['NL', 'PEI', 'NS', 'NB', 'QC', 'ON', 'MB', 'SK', 'AL', 'BC', 'YK', 'NWT', 'NU']
cols = ['Commodity', 'Value ($)']

for province in provinces:
    # Process Global data
    global_df = pd.read_csv(f'{EXPORT_DIR}/{province}-Global.csv', usecols=cols)
    global_df['HS_Code_6digit'] = global_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    global_df[f'{province}_Global'] = global_df['Value ($)']
    global_df = global_df[['HS_Code_6digit', f'{province}_Global']]
    
    tariff_exp_val = tariff_exp_val.merge(global_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_Global'] = tariff_exp_val[f'{province}_Global'] / tariff_exp_val['count']
    
    # Process US data
    us_df = pd.read_csv(f'{EXPORT_DIR}/{province}-US.csv', usecols=cols)
    us_df['HS_Code_6digit'] = us_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    us_df[f'{province}_US'] = us_df['Value ($)']
    us_df = us_df[['HS_Code_6digit', f'{province}_US']]
    
    tariff_exp_val = tariff_exp_val.merge(us_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_US_raw'] = tariff_exp_val[f'{province}_US'] / tariff_exp_val['count']
    ## NOW HAS THE NON UTIL RATE AS WELL!
    tariff_exp_val[f'{province}_US'] = tariff_exp_val[f'{province}_US_raw'] * tariff_exp_val['nonutil_rate']

# Drop the non-relevant columns
tariff_exp_val = tariff_exp_val.drop(columns=['HS_Code_2digit', 'count', 'nonutil_rate'])

tariff_exp_val = tariff_exp_val.fillna(0)

tariff_exp_val

In [ ]:
# Step 1: Aggregate ONLY the necessary sums (Global/US columns)
aggregates = {
    **{f'{province}_Global': (f'{province}_Global', 'sum') for province in provinces},
    **{f'{province}_US': (f'{province}_US', 'sum') for province in provinces},
}

weight_naics = tariff_exp_val.groupby('NAICS', as_index=False).agg(**aggregates)

# Step 2: Compute rates and keep ONLY those columns
for province in provinces:
    global_col = f'{province}_Global'
    us_col = f'{province}_US'
    rate_col = f'{province}'
    
    weight_naics[rate_col] = (
        weight_naics[us_col] / weight_naics[global_col]
    )

# Step 3: Select only naics, naics_2digit, and rate columns
final_columns = ['NAICS'] + [f'{province}' for province in provinces]
weight_naics = weight_naics[final_columns]
weight_naics = weight_naics.fillna(0)

# Result
weight_naics

In [ ]:
# Categories in force BEFORE Aug 22, 2026 (exact strings from CATEGORY_CONFIG)
IN_FORCE_BEFORE = [
    'Auto', 'Aluminum', 'Steel', 'Copper', 'Energy Mineral', 'MHDV',
    'Lumber (old)', 'Lumber (new)', 'before August 22',
]

# An HS code keeps rate 1.0 if ANY of its tags was in force before Aug 22.
# Otherwise the whole code falls back to its chapter non-utilisation rate.
# Masking on HS_Code_6digit (not on the row's own Category) is essential: one HS
# code has several rows, one per tag, each carrying value/count. Masking per row
# would give a single good two different rates and undercount it.
hs_in_force = set(
    tariff_exp_val.loc[tariff_exp_val['Category'].isin(IN_FORCE_BEFORE), 'HS_Code_6digit']
)

t = tariff_exp_val.copy()
rate_sb = np.where(t['HS_Code_6digit'].isin(hs_in_force), 1.0, t['nonutil_chapter'])

for province in provinces:
    t[f'{province}_US'] = t[f'{province}_US_raw'] * rate_sb

weight_naics_sb = t.groupby('NAICS', as_index=False).agg(**aggregates)
for province in provinces:
    weight_naics_sb[province] = weight_naics_sb[f'{province}_US'] / weight_naics_sb[f'{province}_Global']
weight_naics_sb = weight_naics_sb[['NAICS'] + provinces].fillna(0)

print(f"demoted to a zero rate: {(rate_sb == 0).sum()} rows")

weight_naics_sb


In [ ]:
# sanity check: ScenBefore rates can only be <= base
sb = weight_naics_sb.set_index('NAICS')['ON']
base = weight_naics.set_index('NAICS')['ON']
d = (sb - base).dropna()
print(f"above base: {(d > 1e-12).sum()}   (must be 0)")
print(f"below base: {(d < -1e-12).sum()}")
print(d[d < -1e-12].sort_values().head(15))


# STEP 5: Using filtered NAICS Codes to filter directly-impacted businesses and estimate directly-impacted employees

In [ ]:
# DYNAMICALLY CREATE NAICS SETS FOR EACH CATEGORY

# Dictionary to store NAICS codes for each category
category_naics = {}

# Create NAICS sets for each configured category
for csv_name, short_name in CATEGORY_CONFIG.items():
    category_naics[short_name] = set(
        naics_imp[naics_imp['Category'] == csv_name]['NAICS'].unique()
    )
    print(f"  {short_name}: {len(category_naics[short_name])} NAICS codes")

# Special categories (always included)
category_naics['CUSMA'] = set(naics_imp[naics_imp['Category'] == 'nonCUSMA']['NAICS'].unique())
category_naics['Total'] = set(naics_imp['NAICS'].unique())

print(f"\n✅ Created {len(category_naics)} NAICS sets")
print(f"   Total unique NAICS codes: {len(category_naics['Total'])}")

# For backward compatibility, also create individual variables (optional)
total_naics = category_naics['Total']


In [ ]:
SCENARIOS = {
    'ScenBefore': ['before August 22', 'nonCUSMA'],
    'ScenAfter':  ['after August 22',  'nonCUSMA'],
}
# if step 1 returned 0, extend each list with the sectoral categories in force

for name, cats in SCENARIOS.items():
    category_naics[name] = set().union(*(
        set(naics_imp.loc[naics_imp['Category'] == c, 'NAICS'].unique()) for c in cats
    ))

In [ ]:

naics_by_category = pd.DataFrame(
    [(cat, naics) for cat, naics_set in category_naics.items() for naics in naics_set],
    columns=['Category', 'NAICS']
)
naics_by_category.to_csv('../outputs/csv/naics_by_category.csv', index=False)

In [ ]:
print(category_naics['ScenAfter'] == category_naics['Total'])

In [ ]:
# ============================================================
# DYNAMICALLY CREATE COLUMN STRUCTURE
# ============================================================

# Base columns (always present)
col_i = ['DA', 'All_Businesses', 'All_Employees']

# Add columns for each category (Business and Employee counts)
all_category_prefixes = list(CATEGORY_CONFIG.values()) + ['CUSMA', 'Total'] + list(SCENARIOS.keys())
for prefix in all_category_prefixes:
    col_i.extend([f'{prefix}_B', f'{prefix}_E'])

# Add individual NAICS codes as column headers for Est_Employees by NAICS
col_i.extend(sorted(total_naics))

business = pd.DataFrame(columns = col_i)

print(f"✅ Created DataFrame with {len(col_i)} columns")
print(f"   Category columns: {[p for p in all_category_prefixes]}")

In [ ]:
# ============================================================
# MAIN PROCESSING LOOP - DYNAMIC CATEGORIES
# ============================================================

chunk_size = 1_000_000
province_code = {10:'NL',11:'PEI',12:'NS',13:'NB',24:'QC',35:'ON',46:'MB',47:'SK',48:'AL',59:'BC',60:'YK',61:'NWT',62:'NU'}

# melting weights in long form to prepare for a vectorized merge
wlong = (
    weight_naics
    .melt(id_vars='NAICS', var_name='Province', value_name='Rate')
)

wlong = wlong.merge(
    weight_naics_sb.melt(id_vars='NAICS', var_name='Province', value_name='Rate_SB'),
    on=['NAICS', 'Province'], how='outer'
)

# loading necessary data
all_cols = pd.read_csv('../input-data/large_size_data/Dec2022_Estabcounts_byDA.csv', encoding='ISO-8859-1', nrows=1).columns
cols_to_keep = [c for c in all_cols if c != 'Without employees']

# suggesting dtypes for performance purposes
dtype_hint = {
    '1-4':'Int64','5-9':'Int64','10-19':'Int64','20-49':'Int64',
    '50-99':'Int64','100-199':'Int64','200-499':'Int64','500 +':'Int64',
    'Total, with employees':'Int64',
}

total_start = time.time()
chunk_num = 0

# Preparing two empty lists
agg_frames = []       # List for weighted amount of businesses and est employees for each tariff
per_naics_frames = [] # List for total amount of employees for each NAICS code in each ADA --> needed for Step 8 later

# Build the list of aggregation columns dynamically
agg_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    agg_cols.extend([f'{prefix}_B', f'{prefix}_E'])

for chunk in pd.read_csv(
        '../input-data/large_size_data/Dec2022_Estabcounts_byDA.csv',
        encoding='ISO-8859-1',
        chunksize=chunk_size,
        usecols=cols_to_keep,
        dtype=dtype_hint,
    ):
    t0 = time.time()
    chunk_num += 1

    # 1) Filter non-relevant rows
    chunk = chunk[~chunk['NAICS'].isin(['Sub-total, classified', 'Unclassified', 'Total'])].copy()

    # 2) Basic transforms (vectorized)
    # keep NAICS 6-digit as string
    chunk['NAICS'] = chunk['NAICS'].astype(str).str[:6]
    chunk['Business_per_NAICS'] = chunk['Total, with employees'].fillna(0)

    # estimate employees (vectorized)
    chunk['Est_Employees'] = (
        chunk['1-4'].fillna(0) * 3  +
        chunk['5-9'].fillna(0) * 7  +
        chunk['10-19'].fillna(0) * 15 +
        chunk['20-49'].fillna(0) * 35 +
        chunk['50-99'].fillna(0) * 75 +
        chunk['100-199'].fillna(0) * 150 +
        chunk['200-499'].fillna(0) * 350 +
        chunk['500 +'].fillna(0) * 550
    )

    # Province lookup
    # if 'DisseminationAre' isn’t numeric, ensure this still works (it uses first two chars)
    chunk['ProvinceCode'] = chunk['DisseminationAre'].astype(str).str[:2].astype(int, errors='ignore')
    chunk['Province'] = pd.Series(chunk['ProvinceCode']).map(province_code)

    # 3) Merge the per-(NAICS, Province) Rate (vectorized, no apply)
    merged = chunk.merge(wlong, how='left', on=['NAICS','Province'])

    # 4) Weighted columns (vectorized)
    merged['Weighted_Business']  = np.ceil(merged['Business_per_NAICS'] * merged['Rate'])
    merged['Weighted_Employees'] = np.ceil(merged['Est_Employees'] * merged['Rate'])

    # 5) DYNAMIC CATEGORY MASKS - Create masks for each category
    s = merged['NAICS']
    wb = merged['Weighted_Business']
    we = merged['Weighted_Employees']

    # 6) the rate with non section 338 weights (original)
    wb_sb = np.ceil(merged['Business_per_NAICS'] * merged['Rate_SB'])
    we_sb = np.ceil(merged['Est_Employees'] * merged['Rate_SB'])

    # Apply masks dynamically for each category
    for prefix, naics_set in category_naics.items():
        is_in_category = s.isin(naics_set) if len(naics_set) else pd.Series(False, index=s.index)
        # new
        b, e = (wb_sb, we_sb) if prefix == 'ScenBefore' else (wb, we)
        merged[f'{prefix}_B'] = np.where(is_in_category, b, 0)
        merged[f'{prefix}_E'] = np.where(is_in_category, e, 0)

    # Always aggregate the unweighted totals too
    merged['All_Businesses'] = merged['Business_per_NAICS']
    merged['All_Employees']  = merged['Est_Employees']

    # 6) Chunk-level aggregation in ONE groupby
    by_da = merged.groupby('DisseminationAre', as_index=False)[agg_cols].sum()

    # 7) Generating the data for second list --> number of jobs per NAICS in each DA
    is_total = s.isin(total_naics)
    per_naics = (
        merged.loc[is_total, ['DisseminationAre','NAICS','Est_Employees']]
        .pivot_table(index='DisseminationAre', columns='NAICS', values='Est_Employees',
                     aggfunc='sum', fill_value=0)
        .reset_index()
    )

    # Saving the data into the two different lists in each chunk
    agg_frames.append(by_da)
    per_naics_frames.append(per_naics)

    print(f"Chunk {chunk_num} processed in {time.time()-t0:.2f} sec")

# Combine all chunks together to form one big dataframe
agg_all = pd.concat(agg_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

per_naics_all = pd.concat(per_naics_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

business = agg_all.merge(per_naics_all, on='DisseminationAre', how='left')

business = business.rename(columns={'DisseminationAre':'DA'})

print(f"\n✅ All chunks processed in {time.time()-total_start:.2f} sec")
print(f"   Processed categories: {list(category_naics.keys())}")

business

# STEP 6: Regrouping filtered data into CSDs

In [ ]:
# Read DA polygons (KEEP geometry for spatial join)
da = gpd.read_file('../input-data/large_size_data/lda_000b21a_e.shp')
da['DA'] = da['DAUID']
da['DADGUID'] = da['DGUID']
da = da[['DA', 'DADGUID', 'geometry']].copy() # geometry here from the shapefile?

# Read CSD polygons (these DGUIDs match your csd_neighbors IDs: 2021A0005...)
csd = gpd.read_file('../../data/census/lcsd000b21a_e/lcsd000b21a_e.shp')
csdc = csd.copy()
csdc['CSDDGUID'] = csdc['DGUID']
csdc = csdc[['CSDDGUID', 'LANDAREA', 'geometry']].copy()

# Ensure both layers are in a projected CRS for spatial operations
da = da.to_crs('EPSG:3347')
csdc = csdc.to_crs('EPSG:3347')

print('DA rows:', len(da), '| CSD rows:', len(csdc))

In [ ]:
# Build DA -> CSD relation via spatial containment (robust; no DGUID parsing)
da_pts = da[['DA', 'DADGUID', 'geometry']].copy()
da_pts['geometry'] = da_pts.geometry.representative_point()

# Spatial join: DA point within CSD polygon
da_relation = gpd.sjoin(
    da_pts,
    csdc[['CSDDGUID', 'LANDAREA', 'geometry']],
    how='left',
    predicate='within'
).drop(columns=['index_right', 'geometry'])  # keep attributes only; drop DA point geometry

# Match downstream expectations
da_relation['DA'] = pd.to_numeric(da_relation['DA'], errors='coerce').astype('Int64')

match_rate = da_relation['CSDDGUID'].notna().mean()
print(f"DA->CSD match rate (spatial): {match_rate:.3%}")

da_relation.head()

In [ ]:
# ============================================================
# AGGREGATE TO CSD LEVEL - DYNAMIC CATEGORIES
# ============================================================

business_merged = (
    business.merge(da_relation, on='DA', how='right')
) 

# Separate numeric columns from geometry
numeric_cols = [col for col in business.columns if col != 'DA']

# Fill NA only for numeric columns, then cast to int64
for col in numeric_cols:
    business_merged[col] = business_merged[col].fillna(0).astype('int64')

# Build aggregation dictionary dynamically
agg_dict = {
    'All_Businesses': ('All_Businesses', 'sum'),
    'All_Employees': ('All_Employees', 'sum'),
    'LANDAREA': ('LANDAREA', 'first')
}

# Add category columns dynamically
for prefix in all_category_prefixes:
    agg_dict[f'{prefix}_B'] = (f'{prefix}_B', 'sum')
    agg_dict[f'{prefix}_E'] = (f'{prefix}_E', 'sum')

# Add dynamic aggregation rules for each NAICS code
for naics in total_naics:
    agg_dict[naics] = (naics, 'sum')

# Perform grouped aggregation
business_grouped = business_merged.groupby('CSDDGUID', as_index=False).agg(**agg_dict)

# Attach original CSD polygons (not simplified) for downstream outputs
business_grouped = business_grouped.merge(csdc[['CSDDGUID', 'geometry']], on='CSDDGUID', how='left')

print(f"✅ Aggregated to {len(business_grouped)} CSDs with {len(all_category_prefixes)} category pairs")

business_grouped

# STEP 7: Processing it into Centroids for Counts and Choropleth for Rates

Since the earlier table shows the *weighted* numbers of directly exposed businesses and employees (by work location) together with *total* number of employees (by work location) for each affected NAICS code, the former is separated from the latter. The former needs to be processed into separate GDFs for centroids (to show counts) and choropleths (to show percentages)

In [ ]:
# ============================================================
# SEPARATE CATEGORY DATA FROM NAICS DATA - DYNAMIC
# ============================================================

# Build excluded columns list dynamically
excluded_cols = ['All_Businesses', 'All_Employees', 'LANDAREA', 'geometry']
for prefix in all_category_prefixes:
    excluded_cols.extend([f'{prefix}_B', f'{prefix}_E'])

business_filter = business_grouped[['CSDDGUID'] + [col for col in business_grouped.columns if col in excluded_cols]].copy()

business_census = business_grouped[[col for col in business_grouped.columns if col not in excluded_cols]]

print(f"✅ business_filter has {len([c for c in excluded_cols if c.endswith('_B')])} category business columns")
print(f"   business_census has {len(business_census.columns)} NAICS columns")

In [ ]:
# ============================================================
# CREATE CENTROIDS - DYNAMIC CATEGORIES
# ============================================================

# Convert to GeoDataFrame for CENTROIDS (will use point geometry)
cent_gdf = gpd.GeoDataFrame(business_filter.copy(), geometry='geometry', crs = 'EPSG:3347')

# Set a point within each polygon for centroid display
cent_gdf = cent_gdf.drop(columns=['All_Businesses', 'All_Employees', 'LANDAREA'])
cent_gdf['geometry'] = cent_gdf.geometry.representative_point()
cent_gdf.set_geometry('geometry', inplace=True)
cent_gdf

In [ ]:
# Create choropleth from business_filter which still has POLYGON geometries
choro_cols = business_filter.copy()

# Calculate percentage rates for each category dynamically
for prefix in all_category_prefixes:
    # {prefix}_1 = % of businesses affected
    # {prefix}_2 = % of employees affected
    choro_cols[f'{prefix}_1'] = (
        choro_cols[f'{prefix}_B'] / choro_cols['All_Businesses']
    )
    choro_cols[f'{prefix}_2'] = (
        choro_cols[f'{prefix}_E'] / choro_cols['All_Employees']
    )

# Select only the columns we need
rate_cols = []
for prefix in all_category_prefixes:
    rate_cols.extend([f'{prefix}_1', f'{prefix}_2'])

choro_cols = choro_cols[['CSDDGUID'] + rate_cols + ['geometry']]

choro_gdf = gpd.GeoDataFrame(choro_cols, geometry='geometry', crs='EPSG:3347')

print(f"✅ Created choropleth with {len(rate_cols)} rate columns")
print(f"   Categories: {all_category_prefixes}")

choro_gdf

# STEP 8: Finding Neighboring CSDs for Job Accessibility Analysis

Since employees often commute across CSD boundaries, we need to find neighboring CSDs to accurately estimate job accessibility from each residential location.

It is likely that while employees live close to their workplace, they do not live in the same CSD as they work in.  
  
StatsCan Census 2021 data shows a huge drop in the number of Canadians who travel more than 15km to their work vis-a-vis those who travel less than that distance to work.  
  
Thus, this cell creates a dictionary where for each CSD, it lists down, including itself, the CSD IDs within a 15km buffer around it (for small CSDs) or CSD IDs that are adjacent to it (for large CSDs). Small CSDs are defined as CSDs with an area less than (15km)^2 = 706 km^2.

In [ ]:
# # Copy from earlier CSD shapefile and ensure it's in projected CRS (EPSG:3347)
# csds = csdc.to_crs("EPSG:3347").copy()

# # Create a column to mark if CSD is "small" (<= 706 km2)
# csds['is_small'] = csds['LANDAREA'] <= 706

# # More aggressive simplification to prevent memory issues
# # Different tolerance for small vs large CSDs
# csds['geometry'] = csds.apply(
#     lambda row: row['geometry'].simplify(1000 if row['is_small'] else 2000, preserve_topology=True),
#     axis=1
# )

# # Build spatial index
# csds_sindex = csds.sindex

# # Prepare empty dictionary
# csd_neighbors = {}

# # Track overall time
# start_time = time.time()

# print(f"Processing {len(csds)} CSDs...")

# # Process with error handling
# for idx, row in tqdm(csds.iterrows(), total=len(csds)):
#     csd_uid = row['CSDDGUID']
#     geom = row['geometry']
#     is_small = row['is_small']
    
#     try:
#         if is_small:
#             # For small CSDs, use a different approach to avoid buffer issues
#             # Get centroid and create a bounding box instead of buffer
#             centroid = geom.centroid
#             x, y = centroid.x, centroid.y
#             # Create a 15km bounding box around centroid
#             bbox = (x - 15000, y - 15000, x + 15000, y + 15000)
            
#             # Spatial index query using bounding box
#             possible_idx = list(csds_sindex.intersection(bbox))
#             candidates = csds.iloc[possible_idx]
            
#             # Filter by actual distance from centroid (more robust than buffer)
#             matches = candidates[
#                 (candidates['CSDDGUID'] == csd_uid) |
#                 (candidates.geometry.centroid.distance(centroid) <= 15000)
#             ]
#         else:
#             # Use adjacency for large CSDs
#             bounds = geom.bounds
#             # Expand bounds to catch touching geometries
#             expanded_bounds = (bounds[0]-500, bounds[1]-500, bounds[2]+500, bounds[3]+500)
            
#             possible_idx = list(csds_sindex.intersection(expanded_bounds))
#             candidates = csds.iloc[possible_idx]
            
#             # Use distance check instead of touches (more reliable)
#             matches = candidates[
#                 (candidates['CSDDGUID'] == csd_uid) | 
#                 (candidates.geometry.distance(geom) < 500)
#             ]
        
#         csd_neighbors[csd_uid] = matches['CSDDGUID'].tolist()
        
#     except Exception as e:
#         # If there's an error, at minimum include the CSD itself
#         print(f"\nWarning: Error processing {csd_uid}: {str(e)}")
#         csd_neighbors[csd_uid] = [csd_uid]
    
#     # Progress update every 500 CSDs
#     if (idx + 1) % 500 == 0:
#         elapsed = time.time() - start_time
#         rate = (idx + 1) / elapsed
#         remaining = (len(csds) - idx - 1) / rate
#         print(f"Processed {idx + 1}/{len(csds)} CSDs | Elapsed: {timedelta(seconds=int(elapsed))} | ETA: {timedelta(seconds=int(remaining))}")
        
#         # Force garbage collection every 500 items
#         gc.collect()

# # Overall timing
# total_elapsed = time.time() - start_time
# print(f"\n✅ Total time taken: {timedelta(seconds=total_elapsed)}")
# print(f"Average time per CSD: {total_elapsed/len(csds):.3f} seconds")

# # Save the dictionary to disk
# with open("csd_neighbors.json", "w") as f:
#     json.dump(csd_neighbors, f)
    
# print(f"Saved neighbors dictionary with {len(csd_neighbors)} CSDs")

# # Verify completeness
# print(f"Total CSDs with neighbors: {len(csd_neighbors)}")
# print(f"Average neighbors per CSD: {sum(len(v) for v in csd_neighbors.values()) / len(csd_neighbors):.1f}")

In [ ]:
with open('csd_neighbors.json') as f:
    csd_neighbors = json.load(f)

The dictionary is then used in conjunction with the *total* count of employees (by work location), as separated in Cell 13 above, to find out the likely number of jobs of each 6-digit NAICS, and total number of jobs, that are 'accessible' from each CSD --> going by the assumption of travel distance made by Canadians to go to work from Census 2021 data

In [ ]:
# Ensure 'CSDDGUID' is the index for fast lookup
business_census_indexed = business_census.set_index('CSDDGUID')

# Debug: Check what's in business_census
print(f"business_census shape: {business_census.shape}")
print(f"business_census columns: {business_census.columns.tolist()}")
print(f"\nSample of business_census:")
print(business_census.head())
print(f"\nSum of all columns:")
print(business_census.sum())

# Normalize CSD IDs in business_census and ensure numeric job columns
business_census = business_census.copy()
business_census['CSDDGUID'] = business_census['CSDDGUID'].astype(str).str.strip()

business_census_indexed = (
    business_census
    .set_index('CSDDGUID')
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
)

# Normalize csd_neighbors IDs (keys + values)
csd_neighbors_norm = {
    str(k).strip(): [str(v).strip() for v in (vals or [])]
    for k, vals in csd_neighbors.items()
}

# Diagnostics: do neighbor IDs overlap business_census IDs?
idx_set = set(business_census_indexed.index)
keys_set = set(csd_neighbors_norm.keys())
vals_set = set(v for vals in csd_neighbors_norm.values() for v in vals)

print("business_census unique CSDs:", len(idx_set))
print("csd_neighbors keys:", len(keys_set))
print("csd_neighbors values (unique):", len(vals_set))
print("Overlap (neighbors keys ∩ business_census):", len(keys_set & idx_set))
print("Overlap (neighbors values ∩ business_census):", len(vals_set & idx_set))

if len(vals_set & idx_set) == 0:
    print("\n⚠️ No overlap found. Examples:")
    print("business_census sample IDs:", list(sorted(idx_set))[:5])
    print("csd_neighbors key sample IDs:", list(sorted(keys_set))[:5])
    print("csd_neighbors value sample IDs:", list(sorted(vals_set))[:5])

# Prepare list to collect results
aggregated_results = []

# Loop through CSD + its neighbors
for csd_id, neighbor_list in tqdm(csd_neighbors_norm.items()):
    # keep only neighbors that exist in business_census
    keep = list(idx_set.intersection(neighbor_list))
    rows = business_census_indexed.loc[keep] if keep else None

    summed = rows.sum() if rows is not None else business_census_indexed.iloc[:0].sum()
    result = summed.to_dict()
    result['CSDDGUID'] = csd_id
    aggregated_results.append(result)

jobs = pd.DataFrame(aggregated_results)

print(f"\njobs shape: {jobs.shape}")
print("Total jobs (sum over all NAICS cols):", jobs.drop(columns='CSDDGUID').to_numpy().sum())

jobs

Calculate the rate of directly exposed jobs (6-digit NAICS) to total jobs in each industry (1-digit NAICS) within each CSD

In [ ]:
jobs_rate = jobs.copy()

cola = [col for col in jobs.columns if col != 'CSDDGUID']

print(f"Number of NAICS columns: {len(cola)}")
print(f"Sample NAICS columns: {cola[:10]}")

# Calculate summed groups by first digit of column name
jobs_rate['Sum1'] = jobs_rate[[col for col in cola if col.startswith('1')]].sum(axis=1)
jobs_rate['Sum2'] = jobs_rate[[col for col in cola if col.startswith('2')]].sum(axis=1)
jobs_rate['Sum3'] = jobs_rate[[col for col in cola if col.startswith('3')]].sum(axis=1)

print(f"\nSum1 range: {jobs_rate['Sum1'].min()} to {jobs_rate['Sum1'].max()}")
print(f"Sum2 range: {jobs_rate['Sum2'].min()} to {jobs_rate['Sum2'].max()}")
print(f"Sum3 range: {jobs_rate['Sum3'].min()} to {jobs_rate['Sum3'].max()}")

# Compute share per column with proper division by zero handling
for col in cola:
    col_rate = f'{col}_R'
    if col.startswith('1'):
        jobs_rate[col_rate] = np.where(
            jobs_rate['Sum1'] > 0,
            jobs_rate[col] / jobs_rate['Sum1'],
            0
        )
    elif col.startswith('2'):
        jobs_rate[col_rate] = np.where(
            jobs_rate['Sum2'] > 0,
            jobs_rate[col] / jobs_rate['Sum2'],
            0
        )
    elif col.startswith('3'):
        jobs_rate[col_rate] = np.where(
            jobs_rate['Sum3'] > 0,
            jobs_rate[col] / jobs_rate['Sum3'],
            0
        )
    else:
        jobs_rate[col_rate] = 0  # fallback in case of unexpected prefix

# Final filtered DataFrame: only CSDDGUID and the *_R columns
rate_cols = [f'{col}_R' for col in cola]
jobs_rate = jobs_rate[['CSDDGUID'] + rate_cols]

jobs_rate = jobs_rate.fillna(0)

# Debug: Check the results
print(f"\nNon-zero rates count:")
for prefix in ['1', '2', '3']:
    prefix_cols = [c for c in rate_cols if c[0] == prefix]
    if prefix_cols:
        non_zero = (jobs_rate[prefix_cols] > 0).sum().sum()
        total = len(prefix_cols) * len(jobs_rate)
        print(f"  {prefix}xx NAICS: {non_zero}/{total} non-zero values")

print(f"\nSample of jobs_rate:")
print(jobs_rate.head())

jobs_rate.to_csv('trail6_view_csd.csv')

jobs_rate

# STEP 9: Applying Jobs Weight to Census Data

Census 2021 Data reports residents' occupational NAICS code at the two-digit level

In [ ]:
# Use the correct CSD-level census file
census = pd.read_csv('../input-data/large_size_data/98-401-X2021005_English_CSV_data.csv', encoding='latin1')

print(f'Census raw shape: {census.shape}')

# Filter to Census subdivision level only
census = census[census['GEO_LEVEL'] == 'Census subdivision'].copy()
print(f'After GEO_LEVEL filter (Census subdivision): {census.shape}')

# Filter by Characteristic IDs (Employment by Industry at 2-digit NAICS)
census = census[census['CHARACTERISTIC_ID'].isin([2259, 2262, 2263, 2266])].copy()
print(f'After CHARACTERISTIC_ID filter: {census.shape}')

# Create CSDDGUID directly from DGUID column
census['CSDDGUID'] = census['DGUID'].astype(str)

# Province extraction from DGUID (characters 10-11, 0-indexed as 9:11)
census['ProvinceCode'] = census['CSDDGUID'].str[9:11].astype(int)
census['Province'] = census['ProvinceCode'].map(province_code)

# Characteristic Name cleanup to get 2-digit NAICS codes
census['CHARACTERISTIC_NAME'] = (
    census['CHARACTERISTIC_NAME']
    .astype(str)
    .str.replace(' ', '', regex=False)
    .str[:2]
)

census = census[['CSDDGUID', 'Province', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']]

# Pivot
census_pivot = census.pivot_table(
    index=['CSDDGUID', 'Province'],
    columns='CHARACTERISTIC_NAME',
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

print(f"Census pivot shape: {census_pivot.shape}")
print(f"Census pivot columns: {census_pivot.columns.tolist()}")
print(f"Sample IDs (Census): {census_pivot['CSDDGUID'].head().tolist()}")

# --- Diagnostics ---
common_ids = set(census_pivot['CSDDGUID']).intersection(set(jobs_rate['CSDDGUID']))
print(f"Overlap between Census and Jobs Rate IDs: {len(common_ids)}")

if len(common_ids) == 0:
    print("⚠️ No overlap found! Check ID formats.")
    print("Sample IDs (Jobs Rate):", jobs_rate['CSDDGUID'].head().tolist())
else:
    print(f"✅ IDs overlap: {len(common_ids)} CSDs matched")

census_pivot.head()

Thus the weights from Step 8 is used to estimate how many employees (by primary residence) are working in industries directly exposed to tariffs

In [ ]:
# Merge on CSDDGUID to align both datasets - use 'left' to keep all census data
merged = census_pivot.merge(jobs_rate, on='CSDDGUID', how='left')

print(f"Merged shape: {merged.shape}")
print(f"Available columns: {merged.columns.tolist()}")
print(f"Sample 'To' values: {merged['To'].head()}")
print(f"To column stats: min={merged['To'].min()}, max={merged['To'].max()}, mean={merged['To'].mean()}")

# Start building the adjusted DataFrame
adjusted_jobs = merged[['CSDDGUID', 'Province', 'To']].copy()

# Compute adjusted values
for col in cola:
    rate_col = f'{col}_R'
    prefix = col[:2]

    if prefix in ['21', '22']:
        source_col = '21'
    elif prefix in ['31', '32', '33']:
        source_col = '31'
    else:
        source_col = prefix

    # Check if both rate and source columns exist
    if rate_col in merged.columns and source_col in merged.columns:
        adjusted_jobs[col] = np.ceil(merged[rate_col].fillna(0) * merged[source_col].fillna(0))
    else:
        # If either column is missing, assume 0
        adjusted_jobs[col] = 0
        if rate_col not in merged.columns:
            print(f"⚠️ Missing rate column: {rate_col}")
        if source_col not in merged.columns:
            print(f"⚠️ Missing source column: {source_col}")

adjusted_jobs['Sum'] = adjusted_jobs.drop(columns=['CSDDGUID', 'Province', 'To']).sum(axis=1)

adjusted_jobs = adjusted_jobs.fillna(0)

adjusted_jobs['Province'] = adjusted_jobs['Province'].mask(
    adjusted_jobs['Province'].isin([0, np.nan]),
    adjusted_jobs['CSDDGUID'].str[9:11].astype(int).map(province_code)
)

print(f"\nAdjusted jobs shape: {adjusted_jobs.shape}")
print(f"Total adjusted jobs sum: {adjusted_jobs['Sum'].sum()}")
print(f"Rows with non-zero To: {(adjusted_jobs['To'] > 0).sum()}")
print(f"Rows with non-zero Sum: {(adjusted_jobs['Sum'] > 0).sum()}")

adjusted_jobs

# STEP 10: Applying Export Weights to the Census Data

Export weights from Step 4 are applied to account for regional differences

In [ ]:
# Step 1: Melt to long format
long_weight = weight_naics.melt(id_vars='NAICS', var_name='Province', value_name='Rate')

long_weight['NAICS'] = long_weight['NAICS'].astype(str) + '_r'

# Step 2: Pivot to wide format
weight_naics_pivot = long_weight.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

weight_naics_pivot

# same pivot, but on the ScenBefore rates
lw_sb = weight_naics_sb.melt(id_vars='NAICS', var_name='Province', value_name='Rate')
lw_sb['NAICS'] = lw_sb['NAICS'].astype(str) + '_r'
weight_naics_pivot_sb = lw_sb.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

weight_naics_pivot_sb


In [ ]:
# ============================================================
# CALCULATE CENSUS BY TARIFFS - DYNAMIC CATEGORIES
# ============================================================

# Step 1: Merge adjusted_jobs with weight_naics_pivot on Province
census_byjobs = adjusted_jobs.merge(weight_naics_pivot, on='Province', how='left')

# Step 2: Multiply each column by its corresponding rate
for col in cola:
    rate_col = f'{col}_r'
    
    census_byjobs[col] = np.ceil(census_byjobs[col] * census_byjobs[rate_col])

# Step 2b: same thing again on the ScenBefore rates
census_byjobs_sb = adjusted_jobs.merge(weight_naics_pivot_sb, on='Province', how='left')
for col in cola:
    census_byjobs_sb[col] = np.ceil(census_byjobs_sb[col] * census_byjobs_sb[f'{col}_r'])

# Step 3: Keep only CSDDGUID and updated values
census_byjobs = census_byjobs[['CSDDGUID', 'To'] + cola]
census_byjobs_sb = census_byjobs_sb[['CSDDGUID', 'To'] + cola]

# Define output dictionary
grouped_data = {
    'CSDDGUID': census_byjobs['CSDDGUID'],  # retain CSD ID
    'Census': census_byjobs['To'],
}

# Add _C columns for each category dynamically
for prefix, naics_set in category_naics.items():
    src = census_byjobs_sb if prefix == 'ScenBefore' else census_byjobs
    matching_cols = [col for col in cola if col in naics_set]
    if matching_cols:
        grouped_data[f'{prefix}_C'] = src[matching_cols].sum(axis=1)
    else:
        grouped_data[f'{prefix}_C'] = 0

# Create final grouped DataFrame
census_bytariffs = pd.DataFrame(grouped_data)

print(f"✅ Created census_bytariffs with {len([k for k in grouped_data if k.endswith('_C')])} category columns")

census_bytariffs

# STEP 11: Processing and adding the data to Choropleth and Centroid GDFs

Add the data on employees (by primary residence) to the GDFs produced in Step 7

In [ ]:
# ============================================================
# MERGE CENSUS DATA WITH CENTROIDS - DYNAMIC CATEGORIES
# ============================================================

centroids = cent_gdf.merge(census_bytariffs, on='CSDDGUID', how='left')

# Build column list dynamically
centroid_cols = ['CSDDGUID']
for prefix in all_category_prefixes:
    centroid_cols.extend([f'{prefix}_B', f'{prefix}_E', f'{prefix}_C'])
centroid_cols.append('geometry')

centroids = centroids[centroid_cols]
centroids = centroids.to_crs('EPSG:4326')
centroids.to_file('../outputs/geojson/centroids_csd.geojson', driver='GeoJSON')
centroids.to_csv("../outputs/csv/centroids_csd.csv", index=False)

print(f"✅ Saved centroids with {len(all_category_prefixes)} categories x 3 metrics (B, E, C)")

centroids

In [ ]:
# ============================================================
# CALCULATE PERCENTAGES FOR CHOROPLETH - DYNAMIC CATEGORIES
# ============================================================

perc_bytariffs = census_bytariffs.copy()

# Calculate _3 (% of census population in affected jobs) for each category
for prefix in all_category_prefixes:
    perc_bytariffs[f'{prefix}_3'] = (
        perc_bytariffs[f'{prefix}_C'] / perc_bytariffs['Census']
    )

# Select only the percentage columns
perc_cols = ['CSDDGUID'] + [f'{prefix}_3' for prefix in all_category_prefixes]
perc_bytariffs = perc_bytariffs[perc_cols]

# # Clip values to max 1 (100%) for CUSMA and Total
# perc_bytariffs['CUSMA_3'] = perc_bytariffs['CUSMA_3'].clip(upper=1)
# perc_bytariffs['Total_3'] = perc_bytariffs['Total_3'].clip(upper=1)

for p in ['CUSMA', 'Total', 'ScenBefore', 'ScenAfter']:
    perc_bytariffs[f'{p}_3'] = perc_bytariffs[f'{p}_3'].clip(upper=1)

print(f"✅ Calculated _3 percentages for {len(all_category_prefixes)} categories")

perc_bytariffs

In [ ]:
# ============================================================
# MERGE ALL DATA INTO FINAL CHOROPLETH - DYNAMIC CATEGORIES
# ============================================================

# Use 'left' merge to preserve all geometries from choro_gdf
choropleth = choro_gdf.merge(perc_bytariffs, on='CSDDGUID', how='left')

print(f"Choropleth shape after merge: {choropleth.shape}")

print(f"Geometry column type: {type(choropleth['geometry'].iloc[0]) if len(choropleth) > 0 else 'Empty'}")

print(f"Non-null geometries: {choropleth['geometry'].notna().sum()}")

print(f"CRS: {choropleth.crs}")

# Build final column list dynamically
final_cols = ['CSDDGUID']
for prefix in all_category_prefixes:
    final_cols.extend([f'{prefix}_1', f'{prefix}_2', f'{prefix}_3'])
final_cols.append('geometry')

choropleth = choropleth[final_cols]
print(f"Final choropleth shape: {choropleth.shape}")
print(f"Columns: {list(choropleth.columns)}")

# Ensure it's still a GeoDataFrame before CRS conversion
if not isinstance(choropleth, gpd.GeoDataFrame):
    print("⚠️ Converting back to GeoDataFrame")
    choropleth = gpd.GeoDataFrame(choropleth, geometry='geometry', crs='EPSG:3347')

# Convert to WGS84 (EPSG:4326) for better compatibility with mapping software
print(f"Converting from {choropleth.crs} to EPSG:4326...")
choropleth = choropleth.to_crs('EPSG:4326')
print(f"✅ CRS after conversion: {choropleth.crs}")

In [ ]:
# Delete old files if they exist to ensure clean save
import os
for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
    try:
        os.remove(f'choropleth_csd{ext}')
    except FileNotFoundError:
        pass

# Verify geometries before saving
print(f"Geometry check before save:")
print(f"  - Total rows: {len(choropleth)}")
print(f"  - Valid geometries: {choropleth.geometry.is_valid.sum()}")
print(f"  - Geometry types: {choropleth.geometry.type.unique()}")
print(f"  - CRS: {choropleth.crs}")

# Save with EPSG:4326 coordinates
choropleth.to_file('../outputs/geojson/choropleth_csd.geojson', driver='GeoJSON')
choropleth.to_file('choropleth_csd.shp', driver='ESRI Shapefile')
print(f"✅ Files saved with CRS: {choropleth.crs}")

In [ ]:
choropleth.head()

In [ ]:
choropleth.drop(columns="geometry").to_csv("../outputs/csv/choropleth_csd.csv", index=False)

# CSV Trails

for double-checking purposes

In [ ]:
business_grouped.drop(columns='geometry').to_csv('trail_csd.csv', index=False)
business_filter.drop(columns='geometry').to_csv('trail2_csd.csv', index=False)
business_census.to_csv('trail3_csd.csv', index=False)

In [ ]:
choro_cols.drop(columns='geometry').to_csv('trail4_csd.csv', index=False)

In [ ]:
jobs.to_csv('trail5_csd.csv')
jobs_rate.to_csv('trail6_csd.csv')
adjusted_jobs.to_csv('trail7_csd.csv')